# 01 - Simulation Basics And Metric Contract

Use this notebook first. It shows the shortest path through the codebase:
`YAML config -> SimulationConfig -> ClinicAppointmentSimulation -> SimulationResults -> shared analysis metrics`.

The goal is to understand what the engine returns and which metric helpers should be used by reports and experiments.

In [21]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_dir(start: Path) -> Path:
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs" / "baseline.yaml").exists() and (
            candidate / "simulation" / "config_loader.py"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook location.")


REPO_DIR = find_repo_dir(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from analysis.metrics import (
    METRIC_DEFINITIONS,
    aggregate_result_row,
    class_result_rows,
    metric_definition_rows,
    outcome_rates_from_result,
    outcome_totals,
    result_metrics_from_result,
)
from analysis.plot_style import (
    BASELINE_COLOR,
    driver_heatmap_cmap,
    driver_line_style,
    plot_driver_line,
)
from simulation.config_loader import load_config
from simulation.engine import ClinicAppointmentSimulation
from simulation.model import ThresholdRule

BASE_CONFIG = load_config(REPO_DIR / "configs" / "baseline.yaml")
pd.options.display.max_columns = 120

## Inspect The Baseline Config

The config object is immutable enough for safe experimentation with `dataclasses.replace`.
Each class owns arrival rate, balking rule, cancellation probability, no-show rule, and value.

In [22]:
config = BASE_CONFIG

config_summary = pd.DataFrame(
    [
        {
            "class_id": class_id,
            "lambda_per_day": params.lambda_per_day,
            "balk_threshold": params.balk_prob.threshold,
            "balk_low": params.balk_prob.low,
            "balk_high": params.balk_prob.high,
            "cancel_prob": params.cancel_prob,
            "no_show_threshold": params.no_show_prob.threshold,
            "no_show_low": params.no_show_prob.low,
            "no_show_high": params.no_show_prob.high,
            "value": params.value,
        }
        for class_id, params in config.classes.items()
    ]
)

display(
    pd.Series(
        {
            "slots_per_day": config.slots_per_day,
            "horizon_days": config.horizon_days,
            "burn_in_days": config.burn_in_days,
            "measure_days": config.measure_days,
            "cooldown_days": config.cooldown_days,
            "seed": config.seed,
        },
        name="baseline",
    ).to_frame()
)
display(config_summary)

,baseline
slots_per_day,32
horizon_days,14
burn_in_days,30
measure_days,365
cooldown_days,10
seed,11


,class_id,lambda_per_day,balk_threshold,balk_low,balk_high,cancel_prob,no_show_threshold,no_show_low,no_show_high,value
0,1,50.0,9,0.0,0.5,0.1,6,0.0,0.3,1.0
1,2,50.0,9,0.0,0.5,0.1,6,0.0,0.3,1.0


## Run One Simulation

`result_metrics_from_result` is the canonical two-class metric contract for the study.
`class_result_rows` and `aggregate_result_row` are the row builders used by sweeps.

In [23]:
result = ClinicAppointmentSimulation(config).run()

metric_table = pd.Series(result_metrics_from_result(result), name="value").to_frame()
class_table = pd.DataFrame(class_result_rows(result))
aggregate_row = pd.Series(aggregate_result_row(result), name="value").to_frame()

display(metric_table)
display(class_table)
display(aggregate_row)

,value
average_utilization,0.833733
overall_percent_serviced,0.264675
mean_accepted_booking_delay,8.518010
mean_offered_booking_delay,9.502149
class_1_percent_serviced,0.264222
class_2_percent_serviced,0.265137
overall_balking_rate,0.360345
class_1_balking_rate,0.358264
class_2_balking_rate,0.362464
class_1_slot_utilization,0.419521


,class_id,arrivals,booked,balked,offered,no_offer,canceled,no_show,served,mean_accepted_booking_delay,mean_offered_booking_delay,percent_serviced,slot_utilization,balking_rate,total_booking_delay,total_offered_booking_delay
0,1,18545,11901,6644,18545,0,6001,995,4900,8.532224,9.509086,0.264222,0.419521,0.358264,101542.0,176346.0
1,2,18217,11614,6603,18217,0,5843,933,4830,8.503444,9.495087,0.265137,0.413527,0.362464,98759.0,172972.0


,value
average_utilization,0.833733
overall_percent_serviced,0.264675
total_served,9730.000000
total_value,10858.000000
mean_accepted_booking_delay,8.518010
mean_offered_booking_delay,9.502149
overall_balking_rate,0.360345
total_arrivals,36762.000000
total_booked,23515.000000
total_offered,36762.000000


## Metric Definitions

These names are the ones the report and notebooks should use consistently.

In [24]:
pd.DataFrame(metric_definition_rows())

,name,label,unit,direction,description
0,average_utilization,Average utilization,rate,higher means more completed slot use,Completed visits divided by available measured...
1,overall_percent_serviced,Overall percent serviced,rate,higher means more measured arrivals reached se...,Served measured arrivals divided by all measur...
2,mean_accepted_booking_delay,Mean accepted booking delay,days,lower means accepted patients booked sooner,Average offered delay among patients who accep...
3,mean_offered_booking_delay,Mean offered booking delay,days,lower means offered appointments were sooner,Average offered delay among patients who recei...
4,overall_balking_rate,Overall balking rate,rate,higher means more offered patients rejected ap...,Balked patients divided by patients who receiv...
5,access_advantage_class_1,Class 1 access advantage,rate difference,positive means Class 1 has a higher served rat...,Class 1 percent serviced minus Class 2 percent...
6,delay_advantage_class_1,Class 1 delay advantage,day difference,positive means Class 1 has lower offered delay...,Class 2 mean offered booking delay minus Class...


## Outcome Rates And Measurement Semantics

`unresolved_booked` is a diagnostic for late measurement-window bookings that have not resolved by simulation end.
The current model documents this caveat but does not change engine behavior.

In [25]:
totals = outcome_totals(result)
rates = outcome_rates_from_result(result)

display(pd.Series(totals, name="count").to_frame())
display(pd.Series(rates, name="rate").to_frame())
print(f"cooldown_days = {config.cooldown_days}; horizon_days - 1 = {config.horizon_days - 1}")

,count
arrivals,36762
booked,23515
balked,13247
no_offer,0
canceled,11844
no_show,1928
served,9730
offered,36762
unresolved_booked,13


,rate
total_arrivals,36762.000000
total_booked,23515.000000
served_rate,0.264675
balked_rate,0.360345
no_offer_rate,0.000000
canceled_rate,0.322181
no_show_rate,0.052445
unresolved_booked_rate,0.000354
lost_after_booking_rate,0.374980


cooldown_days = 10; horizon_days - 1 = 13


## Inspect Calendar State

The final calendar stores one row per residual day and one entry per daily slot.
A tuple `(class_id, tau)` means a booked patient and their original offered delay.

In [26]:
final_calendar = pd.DataFrame(result.final_full_state)
print(f"final calendar shape: {final_calendar.shape}")
display(final_calendar.T)

print("first two start-of-day summary states:")
for state in result.daily_summary_states[:2]:
    display(pd.DataFrame(state).T)

final calendar shape: (14, 32)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,"(2, 11)","(2, 11)","(1, 11)","(2, 11)","(2, 12)","(1, 11)","(2, 12)","(2, 11)","(2, 11)","(2, 11)","(2, 11)","(1, 11)",0,0
1,"(1, 11)","(1, 11)","(2, 11)","(2, 11)","(2, 12)","(2, 11)","(1, 12)","(1, 11)","(1, 11)","(2, 11)","(2, 11)","(2, 11)",0,0
2,"(1, 11)","(1, 11)","(2, 11)","(2, 11)","(1, 11)","(2, 11)","(2, 12)","(1, 11)","(2, 11)","(2, 11)","(1, 11)","(1, 11)",0,0
3,"(1, 11)","(2, 11)","(1, 11)","(2, 11)","(1, 11)","(2, 11)","(1, 11)","(1, 11)","(2, 11)","(2, 11)","(1, 11)","(1, 11)",0,0
4,"(1, 11)","(2, 11)","(2, 11)","(1, 11)","(2, 11)","(2, 11)","(1, 11)","(1, 11)","(2, 11)","(1, 11)","(2, 11)","(1, 11)",0,0
5,"(2, 11)","(1, 11)","(1, 11)","(2, 11)","(2, 11)","(2, 11)","(2, 11)","(1, 11)","(1, 11)","(2, 11)","(2, 11)","(1, 11)",0,0
6,"(2, 11)","(2, 11)","(1, 11)","(2, 11)","(1, 11)","(1, 11)","(1, 11)","(2, 11)","(1, 11)","(2, 11)","(2, 11)","(2, 11)",0,0
7,"(1, 11)","(2, 11)","(1, 11)","(2, 11)","(1, 11)","(1, 11)","(1, 11)","(2, 11)","(2, 11)","(1, 11)","(1, 11)","(1, 11)",0,0
8,"(2, 11)","(1, 11)","(1, 11)","(2, 11)","(2, 11)","(2, 11)","(1, 11)","(1, 11)","(1, 11)","(1, 11)","(2, 11)","(2, 11)",0,0
9,"(2, 10)","(2, 11)","(2, 11)","(1, 11)","(1, 11)","(1, 11)","(1, 11)","(2, 11)","(1, 11)","(2, 11)","(1, 11)","(1, 11)",0,0


first two start-of-day summary states:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
1,14,14,19,16,17,16,20,12,8,14,16,0,0,0
2,18,18,12,11,12,14,10,17,23,14,8,0,0,0


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
1,14,19,17,16,15,19,13,8,13,19,12,0,0,0
2,18,12,14,12,15,11,16,23,15,10,13,0,0,0


## Small Seed Check

Use this cell to see simulation noise before running larger sweeps.

In [27]:
config

SimulationConfig(slots_per_day=32, horizon_days=14, burn_in_days=30, measure_days=365, cooldown_days=10, classes={1: PatientClassParams(class_id=1, lambda_per_day=50.0, balk_prob=ThresholdRule(threshold=9, low=0.0, high=0.5), cancel_prob=0.1, no_show_prob=ThresholdRule(threshold=6, low=0.0, high=0.3), value=1.0), 2: PatientClassParams(class_id=2, lambda_per_day=50.0, balk_prob=ThresholdRule(threshold=9, low=0.0, high=0.5), cancel_prob=0.1, no_show_prob=ThresholdRule(threshold=6, low=0.0, high=0.3), value=1.0)}, seed=11)

In [28]:
seed_rows = []
for seed in [11, 12, 13, 14, 15]:
    seeded_config = replace(config, seed=seed)
    seeded_result = ClinicAppointmentSimulation(seeded_config).run()
    seed_rows.append({"seed": seed, **result_metrics_from_result(seeded_result)})

seed_df = pd.DataFrame(seed_rows)
display(seed_df[["seed", "average_utilization", "overall_percent_serviced", "mean_offered_booking_delay", "overall_balking_rate", "access_advantage_class_1"]])
seed_df.describe(include='number').T

,seed,average_utilization,overall_percent_serviced,mean_offered_booking_delay,overall_balking_rate,access_advantage_class_1
0,11,0.833733,0.264675,9.502149,0.360345,-0.000915
1,12,0.845377,0.272343,9.307673,0.353961,-0.001193
2,13,0.845719,0.271152,9.231608,0.358094,-0.001633
3,14,0.840154,0.268364,9.310965,0.357235,0.001155
4,15,0.840668,0.268023,9.312170,0.351427,-0.001873


,count,mean,std,min,25%,50%,75%,max
seed,5.0,13.000000,1.581139,11.000000,12.000000,13.000000,14.000000,15.000000
average_utilization,5.0,0.841130,0.004873,0.833733,0.840154,0.840668,0.845377,0.845719
overall_percent_serviced,5.0,0.268912,0.002993,0.264675,0.268023,0.268364,0.271152,0.272343
mean_accepted_booking_delay,5.0,8.365250,0.091718,8.270075,8.336116,8.350209,8.351839,8.518010
mean_offered_booking_delay,5.0,9.332913,0.100564,9.231608,9.307673,9.310965,9.312170,9.502149
class_1_percent_serviced,5.0,0.268466,0.002932,0.264222,0.267083,0.268941,0.270338,0.271744
class_2_percent_serviced,5.0,0.269357,0.003166,0.265137,0.267786,0.268957,0.271971,0.272937
overall_balking_rate,5.0,0.356212,0.003522,0.351427,0.353961,0.357235,0.358094,0.360345
class_1_balking_rate,5.0,0.355042,0.002264,0.352035,0.354210,0.355034,0.355669,0.358264
class_2_balking_rate,5.0,0.357401,0.005490,0.350825,0.352269,0.359443,0.362003,0.362464
